# ARTICLE2_NB4: Combined run (all eight faults) and the diagnostic checks C1-C7

**This notebook produces results for the fault-injection study (see README).**

Protocol: *study protocol* (frozen 11 Nov 2026), Section 3 ("Combined run"), Section 4 (outcomes) and Section 6 (checks C1-C7).

**What it does**
- **CELL 2 - ARFIMA models.** Re-estimates the reference ARFIMA and the F1 route, and builds the F1 + F2 ARFIMA used by the combined run.
- **CELL 3 - Combined run.** All eight faults together, five seeds. No network is retrained: LSTM comes from the NB2 reference cache, XGBoost is refitted with the leaked column, baselines come from the NB3 cache. Compared in words with the headline claims of v1. This is **not** an exact reproduction of v1 (library versions differ).
- **CELLS 4-5 - Checks C1-C7.** Runs the seven checks on the reference, on each fault alone and on the combined run (seed S0), and builds the detection matrix. The reference must pass all seven.
- **CELL 6 - Significance counts.** Unadjusted counts for comparison with the first article.

**Code:** `03_Code/article2_diagnostics.py` (checks and combined run). It imports the reference code and never edits it.
**Inputs and outputs:** reads only `Article 2/02_Inputs_v2_snapshot` and `Article 2/05_Outputs`; writes only `Article 2/05_Outputs/combined` and `Article 2/05_Outputs/diagnostics`.
**Runtime:** about 10-15 minutes on a T4. Run NB1, NB2 and NB3 first.

## CELL 0: Environment setup (Article 2 folder only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

%cd "/content/drive/MyDrive/Article 2"

import os, sys
from pathlib import Path
ROOT = Path(os.getcwd())
assert ROOT.name == "Article 2", f"Wrong folder: {ROOT}"
CODE_DIR = ROOT / "03_Code"
for f in ["article2_pipeline.py", "article2_faults.py", "article2_tests.py", "article2_baselines.py", "article2_diagnostics.py"]:
    assert (CODE_DIR / f).exists(), f"{f} missing from {CODE_DIR}"
sys.path.insert(0, str(CODE_DIR))
print("Working directory:", ROOT)

## CELL 1: Imports, settings and the saved results of NB1-NB3

In [ ]:
import json, time, pickle, importlib, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
import sklearn, scipy, statsmodels
warnings.filterwarnings("ignore")

import article2_pipeline as ap
import article2_faults as af
import article2_tests as at
import article2_baselines as ab
import article2_diagnostics as dg
for m in (ap, af, at, ab, dg):
    importlib.reload(m)

P = ap.paths(ROOT)
OUT = P["OUT"]
REF_DIR, FDIR, BDIR = OUT / "reference", OUT / "faults", OUT / "baselines"
CDIR, DDIR = OUT / "combined", OUT / "diagnostics"
DCACHE = DDIR / "_cache"
for d in [CDIR, DDIR, DCACHE, OUT / "logs"]:
    d.mkdir(parents=True, exist_ok=True)
SEEDS = {"S0": 42, "S1": 101, "S2": 202, "S3": 303, "S4": 404}

need = [REF_DIR / "reference_seed_range.csv", FDIR / "reference_metrics_long.csv", FDIR / "reference_dm_sets.csv"]
need += [FDIR / "_reference_cache" / f"seed_{s}.pkl" for s in SEEDS.values()]
need += [BDIR / "_cache" / f"seed_{s}.pkl" for s in SEEDS.values()]
need += [FDIR / f / "seed_42" / "JPM_test_all_models.csv" for f in af.MODEL_FAULTS]
missing = [str(p.relative_to(ROOT)) for p in need if not p.exists()]
assert not missing, f"Run NB1-NB3 first. Missing: {missing}"

VERSIONS = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
            "tensorflow": tf.__version__, "xgboost": xgb.__version__, "scikit-learn": sklearn.__version__,
            "scipy": scipy.__version__, "statsmodels": statsmodels.__version__,
            "gpu": [d.name for d in tf.config.list_physical_devices("GPU")],
            "run_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
for k, v in VERSIONS.items():
    print(f"{k:14s}: {v}")
print("Input snapshot:", ap.check_input_snapshot(P))

def read_tables(folder):
    return {t: pd.read_csv(folder / f"{t}_test_all_models.csv", parse_dates=["Date"], index_col="Date") for t in ap.TICKERS}

data = ap.load_data(P)
arfima_snap = ap.load_arfima(P)
REFB = {}
for s in SEEDS.values():
    with open(FDIR / "_reference_cache" / f"seed_{s}.pkl", "rb") as f:
        REFB[s] = dict(zip(["base", "serial"], pickle.load(f)))
BASE = {}
for s in SEEDS.values():
    with open(BDIR / "_cache" / f"seed_{s}.pkl", "rb") as f:
        BASE[s] = pickle.load(f)
REFC = {s: read_tables(REF_DIR / f"seed_{s}") for s in SEEDS.values()}
FAULTC = {f: read_tables(FDIR / f / "seed_42") for f in af.MODEL_FAULTS}
ref_metrics = pd.read_csv(FDIR / "reference_metrics_long.csv")
ref_dm = pd.read_csv(FDIR / "reference_dm_sets.csv")
scalers = af.minmax_scalers(data)
print("NB1-NB3 results loaded")

## CELL 2: ARFIMA models (about 3 minutes, CPU; cached)

Reference route (v2) and F1 route (v1, silent failures). The combined run uses the F1 model with the static forecast of F2.

In [ ]:
afile = DCACHE / "arfima_models.pkl"
if afile.exists():
    with open(afile, "rb") as f:
        ar_ref, ar_ref_models, ar_f1, ar_f1_models, ar_logs = pickle.load(f)
    print("Loaded from cache")
else:
    t0 = time.time()
    ar_ref, ar_ref_models, l1 = af.fit_arfima(data)
    ar_f1, ar_f1_models, l2 = af.fit_arfima(data, fault="F1")
    ar_logs = pd.concat([l1, l2], ignore_index=True)
    with open(afile, "wb") as f:
        pickle.dump((ar_ref, ar_ref_models, ar_f1, ar_f1_models, ar_logs), f)
    print(f"Estimated in {time.time() - t0:.0f} s")
ar_f2 = af.static_arfima_tables(data, ar_ref_models)         # F2 alone (reference model, static forecast)
ar_comb = af.static_arfima_tables(data, ar_f1_models)        # F1 + F2 together (combined run)
print(ar_logs[["Ticker", "Fault", "p", "d", "q", "AIC", "Fits_logged", "Fits_dropped_silently"]].round(2).to_string(index=False))

## CELL 3: Combined run - all eight faults, five seeds (about 1-2 minutes)

F1 + F2 (ARFIMA), F3 (units), F4 (leaked XGBoost), F5 (placeholder), F6 (pseudo-residual Serial), F7 (testing), F8 (baselines). Outcomes as in NB2, then the v1 headline claims next to the combined run and the reference.

In [ ]:
COMB, comb_logs, comb_base, comb_ens = {}, [], {}, {}
cm, cdm, f7c, rank_rows = [], [], [], []
for label, seed in SEEDS.items():
    comb, logs, base_l, ens = dg.run_combined(data, REFB[seed]["base"], ar_comb, seed, scalers)
    COMB[seed], comb_base[seed], comb_ens[seed] = comb, base_l, ens
    d = CDIR / f"seed_{seed}"
    d.mkdir(exist_ok=True)
    for t in ap.TICKERS:
        comb[t].to_csv(d / f"{t}_test_all_models.csv")
    for k, v in logs.items():
        comb_logs.append(v.assign(Log=k))
    cm.append(af.metrics_long(comb, "COMBINED", seed))
    cdm.append(af.dm_sets(comb).assign(Run="COMBINED", Seed=seed))
    fam = at.f7_family(comb)
    f7c.append({"Seed": seed, "F7_significant_of_84": int(fam.Sig_F7.sum()),
                "Share_p_below_0.001": float((fam.p_plain < 0.001).mean()),
                "Ref_protocol_significant_of_9": int(ap.primary_dm(REFC[seed]).Sig_Holm.sum())})
    # ranking: combined research models + F8 baselines vs reference research + reference baselines
    r_c = pd.concat([ab.research_metrics(comb, seed), ab.baseline_metrics(BASE[seed]["f8"], "F8", seed)])
    r_r = pd.concat([ab.research_metrics(REFC[seed], seed), ab.baseline_metrics(BASE[seed]["ref"], "REF", seed)])
    rc, rr = r_c.groupby("Model").MAE_ratio.mean(), r_r.groupby("Model").MAE_ratio.mean()
    rc["Persistence"] = rr["Persistence"] = 1.0
    rc = rc.loc[rr.index]
    arch = ["Serial (Final)", "Parallel Equal", "Parallel InvError", "Parallel SoftMax", "Stacking"]
    rank_rows.append({"Seed": seed, "Tau": float(ap.stats.kendalltau(rr.round(3).values, rc.round(3).values).statistic),
                      "Best_architecture_ratio_REF": float(rr[arch].min()), "Best_baseline_ratio_REF": float(rr[ab.BASELINES].min()),
                      "Best_architecture_ratio_COMB": float(rc[arch].min()), "Best_baseline_ratio_COMB": float(rc[ab.BASELINES].min())})
cm = pd.concat(cm, ignore_index=True)
cdm = pd.concat(cdm, ignore_index=True)
comb_logs = pd.concat(comb_logs, ignore_index=True)
f7c, rank_c = pd.DataFrame(f7c), pd.DataFrame(rank_rows)
cm.to_csv(CDIR / "combined_metrics_long.csv", index=False)
cdm.to_csv(CDIR / "combined_dm_sets.csv", index=False)
comb_logs.to_csv(CDIR / "combined_logs.csv", index=False)
f7c.to_csv(CDIR / "combined_f7.csv", index=False)
rank_c.to_csv(CDIR / "combined_ranking.csv", index=False)

# Delta MAE ratio and materiality (Protocol Section 4)
m = cm.merge(ref_metrics[["Seed", "Ticker", "Model", "MAE_ratio"]], on=["Seed", "Ticker", "Model"], suffixes=("", "_ref"))
m = m[m.Model != "Persistence"]
m["Delta"] = m.MAE_ratio - m.MAE_ratio_ref
mean_stock = m.groupby(["Seed", "Model"])[["MAE_ratio", "MAE_ratio_ref", "Delta"]].mean().reset_index().assign(Ticker="Mean")
m = pd.concat([m, mean_stock], ignore_index=True)
summ = m.groupby(["Model", "Ticker"]).agg(Comb_ratio=("MAE_ratio", "mean"), Ref_ratio=("MAE_ratio_ref", "mean"),
                                          Delta_mean=("Delta", "mean"), Delta_min=("Delta", "min"), Delta_max=("Delta", "max")).reset_index()
sr = pd.read_csv(REF_DIR / "reference_seed_range.csv").rename(columns={"Stock": "Ticker", "min": "Ref_min", "max": "Ref_max"})
summ = summ.merge(sr[["Model", "Ticker", "Ref_min", "Ref_max"]], on=["Model", "Ticker"], how="left")
summ["Outside_ref_range"] = (summ.Comb_ratio.round(3) < summ.Ref_min.round(3)) | (summ.Comb_ratio.round(3) > summ.Ref_max.round(3))
summ.to_csv(CDIR / "combined_delta_summary.csv", index=False)

pd.set_option("display.width", 220)
print("COMBINED RUN - MAE ratio to persistence (mean over seeds): reference -> combined")
tab = summ.copy()
tab["Cell"] = tab.apply(lambda r: f"{r.Ref_ratio:.3f} -> {r.Comb_ratio:.3f}", axis=1)
print(tab.pivot(index="Model", columns="Ticker", values="Cell").loc[af.ALL_MODELS, ap.TICKERS + ["Mean"]].to_string())
print()
sig = cdm.groupby(["Seed", "Set"]).Sig_Holm.sum().unstack().mean()
print("Significance counts after Holm (mean over seeds):", sig.round(1).to_dict())
print("F7 on the combined run: significant of 84 =", f7c.F7_significant_of_84.tolist(),
      "| share with p < 0.001 =", f7c["Share_p_below_0.001"].round(3).tolist())
print()
print("Architectures vs baselines (combined uses the F8 baselines):")
print(rank_c.round(3).to_string(index=False))

In [ ]:
# The v1 headline claims next to the combined run and the reference (mean over seeds)
def per_stock(metrics, model, col="MAE"):
    return metrics[metrics.Model == model].groupby("Ticker")[col].mean()

def pct_vs(metrics, a, b="LSTM"):
    ma, mb = per_stock(metrics, a), per_stock(metrics, b)
    return 100 * (mb - ma) / mb                     # positive = a better than b

def fmt(s, d=1, unit="%"):
    return ", ".join(f"{t} {s[t]:+.{d}f}{unit}" if unit == "%" else f"{t} {s[t]:.{d}f}" for t in ap.TICKERS)

refm = ref_metrics.copy()
w_comb = comb_logs[comb_logs.Log == "weights"].groupby(["Ticker", "Scheme"])[["w_ARFIMA", "w_LSTM", "w_XGB"]].mean()
c_comb = comb_logs[comb_logs.Log == "stacking"].groupby("Ticker")[["b0", "b_ARFIMA", "b_LSTM", "b_XGB"]].mean()
d_f1 = ar_logs[ar_logs.Fault == "F1"].set_index("Ticker")["d"]
d_ref = ar_logs[ar_logs.Fault == "REF"].set_index("Ticker")["d"]
claims = [
    ("ARFIMA d collapsed to 0 for all three stocks", fmt(d_f1, 1, ""), fmt(d_ref, 1, "")),
    ("ARFIMA standalone test R2 from -0.69 to -4.33", fmt(per_stock(cm, "ARFIMA", "R2_price"), 2, ""), fmt(per_stock(refm, "ARFIMA", "R2_price"), 3, "")),
    ("Stacking MAE vs LSTM: JPM +35%, XOM +65%, NEE -186%", fmt(pct_vs(cm, "Stacking")), fmt(pct_vs(refm, "Stacking"))),
    ("Stacking test R2: JPM 0.809, XOM -1.79, NEE -0.62", fmt(per_stock(cm, "Stacking", "R2_price"), 3, ""), fmt(per_stock(refm, "Stacking", "R2_price"), 3, "")),
    ("Parallel InvError: 7-15% better (vs LSTM) on all stocks", fmt(pct_vs(cm, "Parallel InvError")), fmt(pct_vs(refm, "Parallel InvError"))),
    ("InvError weights: ARFIMA = LSTM, XGBoost near zero",
     "; ".join(f"{t} {w_comb.loc[(t, 'InvError')].round(3).tolist()}" for t in ap.TICKERS), "about 1/3 each (first article)"),
    ("Stacking: XGBoost coef near 0, LSTM 1.09-1.93, negative intercept",
     "; ".join(f"{t} b0 {c_comb.loc[t, 'b0']:.2f}, LSTM {c_comb.loc[t, 'b_LSTM']:.2f}, XGB {c_comb.loc[t, 'b_XGB']:.3f}" for t in ap.TICKERS),
     "ARFIMA coef 0; JPM 0.99 x LSTM + 1.16 (first article)"),
    ("Serial Stage 3 catastrophic: MAE 110/116, R2 -63,218", fmt(per_stock(cm, "Serial (Final)"), 1, "") + " | R2 " + fmt(per_stock(cm, "Serial (Final)", "R2_price"), 0, ""),
     fmt(per_stock(refm, "Serial (Final)"), 3, "")),
    ("All DM comparisons p < 0.001", f"{f7c.F7_significant_of_84.mean():.1f} of 84 significant; {100 * f7c['Share_p_below_0.001'].mean():.0f}% with p < 0.001",
     "2 of 9 significant (Holm)"),
    ("Order-of-magnitude improvement over baselines", f"best architecture {rank_c.Best_architecture_ratio_COMB.mean():.3f} vs best baseline {rank_c.Best_baseline_ratio_COMB.mean():.3f}",
     f"best architecture {rank_c.Best_architecture_ratio_REF.mean():.3f} vs best baseline {rank_c.Best_baseline_ratio_REF.mean():.3f}"),
]
claims_df = pd.DataFrame(claims, columns=["Earlier version (v1) claim", "Combined run (all eight faults)", "Reference (v2)"])
claims_df.to_csv(CDIR / "v1_claims_vs_combined.csv", index=False)
for _, r in claims_df.iterrows():
    print(f"- {r['Earlier version (v1) claim']}\n    combined:  {r['Combined run (all eight faults)']}\n    reference: {r['Reference (v2)']}")
print()
print("Note: ensembles in v1 were scored in MinMax units; the combined run does the same (F3). Directions are compared, not exact values.")

## CELL 4: Models kept for the forecast-origin audit (C6), seed S0 (about 1 minute on a T4)

LSTM and XGBoost are retrained for seed 42 with the models kept, and checked against the cached reference: they must be identical. The F4 XGBoost (with the leaked column) is kept the same way.

In [ ]:
base_cap, models_cap = dg.train_base_capture(data, 42)
cap_diff = max(float(np.max(np.abs(base_cap[t][["LSTM", "XGB"]].values - REFB[42]["base"][t][["LSTM", "XGB"]].values)))
               for t in ap.TICKERS)
print(f"Captured models vs cached reference (seed 42): max difference {cap_diff:.2e} USD")
CAPTURE_EXACT = cap_diff < 1e-6
leak_models = dg.xgb_leak_capture(data, 42)
xgb_ref_models = {t: models_cap[t]["xgb"] for t in ap.TICKERS}
f4_base, _ = af.xgb_base(data, REFB[42]["base"], 42, leak=True)
print("Captured:", CAPTURE_EXACT)

## CELL 5: The seven checks on every run (seed S0) and the detection matrix

Rows: the reference, each fault alone, the combined run. A fail on an expected fault is marked "(expected)"; an expected fault that passes is marked "MISSED". **The reference must pass all seven.**

In [ ]:
TAGS = {c: "USD" for c in ["y_true", "Persistence"] + af.ALL_MODELS}     # what every run declares (faults are silent)
REF_SPEC = dict(dg.REFERENCE_SPEC)
F7_SPEC = dict(REF_SPEC, hac=False, hln=False, n_tests=84, correction=None, winner_from_test_loss=False)
F8_SPEC = dict(REF_SPEC, baseline_train_window="2015-2022", baseline_horizon="recursive 501-step")
COMB_SPEC = dict(F7_SPEC, baseline_train_window="2015-2022", baseline_horizon="recursive 501-step")

def grid_log(models):
    return {t: {"logged": len(models[t]["grid"]), "expected": len(af.D_GRID) * len(af.P_RANGE) * len(af.Q_RANGE),
                "unreported_failures": 0} for t in ap.TICKERS}

def serial_block(comb, trained):
    return {"stage2_log": trained, "stage3_log": trained,
            "outputs": {t: (comb[t]["Serial (ARFIMA+LSTM)"].values, comb[t]["LSTM"].values,
                            comb[t]["Serial (Final)"].values, comb[t]["XGBoost"].values) for t in ap.TICKERS}}

def record(name, comb, arfima_tabs, ar_models, base_tabs, ens, baselines, xgb_models, leak, spec, serial_trained):
    return {"name": name, "arfima_grid_log": grid_log(ar_models),
            "train_forecasts": dg.train_forecasts(arfima_tabs, base_tabs, baselines, data),
            "test_ratios": dg.test_ratios(comb, baselines), "unit_tags": TAGS,
            "component_pairs": dg.ens_pairs(ens, comb), "serial": serial_block(comb, serial_trained),
            "c6_audit": dg.c6_audit(data, comb, ar_models, models_cap, xgb_models, leak=leak), "spec": spec}

b_ref, b_f8 = BASE[42]["ref"], BASE[42]["f8"]
base42 = REFB[42]["base"]
ens = lambda base, ar, faults=(): dg.ens_inputs(base, ar, faults, scalers)
RUNS = {
    "REF": record("REF", REFC[42], arfima_snap, ar_ref_models, base42, ens(base42, arfima_snap), b_ref, xgb_ref_models, False, REF_SPEC, True),
    "F1": record("F1", FAULTC["F1"], ar_f1, ar_f1_models, base42, ens(base42, ar_f1), b_ref, xgb_ref_models, False, REF_SPEC, True),
    "F2": record("F2", FAULTC["F2"], ar_f2, ar_ref_models, base42, ens(base42, ar_f2), b_ref, xgb_ref_models, False, REF_SPEC, True),
    "F3": record("F3", FAULTC["F3"], arfima_snap, ar_ref_models, base42, ens(base42, arfima_snap, ("F3",)), b_ref, xgb_ref_models, False, REF_SPEC, True),
    "F4": record("F4", FAULTC["F4"], arfima_snap, ar_ref_models, f4_base, ens(f4_base, arfima_snap), b_ref, leak_models, True, REF_SPEC, True),
    "F5": record("F5", FAULTC["F5"], arfima_snap, ar_ref_models, base42, ens(base42, arfima_snap, ("F5",)), b_ref, xgb_ref_models, False, REF_SPEC, True),
    "F6": record("F6", FAULTC["F6"], arfima_snap, ar_ref_models, base42, ens(base42, arfima_snap), b_ref, xgb_ref_models, False, REF_SPEC, False),
    "F7": record("F7", REFC[42], arfima_snap, ar_ref_models, base42, ens(base42, arfima_snap), b_ref, xgb_ref_models, False, F7_SPEC, True),
    "F8": record("F8", REFC[42], arfima_snap, ar_ref_models, base42, ens(base42, arfima_snap), b_f8, xgb_ref_models, False, F8_SPEC, True),
    "COMBINED": record("COMBINED", COMB[42], ar_comb, ar_f1_models, comb_base[42], comb_ens[42], b_f8, leak_models, True, COMB_SPEC, False),
}
RESULTS, EXTRA = {}, {}
for name, rec in RUNS.items():
    RESULTS[name], EXTRA[name] = dg.run_checks(rec)
matrix = dg.detection_matrix(RESULTS)
matrix.to_csv(DDIR / "detection_matrix.csv")
reasons = pd.DataFrame([{"Run": r, "Check": c, "Result": "FAIL" if RESULTS[r][c][0] else "pass", "Reason": RESULTS[r][c][1]}
                        for r in RESULTS for c in dg.CHECKS])
reasons.to_csv(DDIR / "check_reasons.csv", index=False)
for name in RUNS:
    EXTRA[name]["C2"].assign(Run=name).to_csv(DDIR / f"C2_training_fit_{name}.csv", index=False)
    EXTRA[name]["C4"].assign(Run=name).to_csv(DDIR / f"C4_component_pairs_{name}.csv", index=False)
    EXTRA[name]["C6"].assign(Run=name).to_csv(DDIR / f"C6_origin_audit_{name}.csv", index=False)

print("DETECTION MATRIX (seed S0)")
print(matrix.to_string())
print()
REF_PASSES_ALL = not any(RESULTS["REF"][c][0] for c in dg.CHECKS)
print("Reference passes all seven checks:", REF_PASSES_ALL)
if not REF_PASSES_ALL:
    print("REFERENCE FAILED:", {c: RESULTS["REF"][c][1] for c in dg.CHECKS if RESULTS["REF"][c][0]})
    print("Per Protocol Section 6 the threshold rule must be fixed in the deviation log. Share this output before using the matrix.")
missed = [(c, f) for c in dg.CHECKS for f in dg.EXPECTED[c] if not RESULTS[f][c][0]]
print("Expected detections missed:", missed or "none")
print()
print("Reasons for every FAIL:")
for _, r in reasons[reasons.Result == "FAIL"].iterrows():
    print(f"  {r.Run:9s} {r.Check}: {r.Reason[:160]}")

## CELL 6: Significance counts - Holm versus unadjusted (comparison with the first article)

In [ ]:
r42 = ref_dm[ref_dm.Seed == 42]
counts = pd.DataFrame({"Holm": r42.groupby("Set").Sig_Holm.sum(), "Unadjusted_5%": r42.groupby("Set")["Sig_5%"].sum()}).astype(int)
print("Reference, seed 42:")
print(counts.to_string())
print("First article reported: primary 2 of 9 (Holm); vs persistence 3 significantly worse; vs ARFIMA 8 in favour of the architecture.")
f7_all = pd.read_csv(OUT / "tests" / "F7_all_84_tests_by_seed.csv")
nw84 = f7_all.assign(Sig_NW_unadj=f7_all.p_HLN < 0.05).groupby("Seed").agg(
    F7_plain_uncorrected=("Sig_F7", "sum"), NW_HLN_uncorrected=("Sig_NW_unadj", "sum"), NW_HLN_Holm=("Sig_ref84", "sum"))
print()
print("The 84 pairs three ways (variance formula vs correction):")
print(nw84.to_string())
counts.to_csv(DDIR / "reference_counts_holm_vs_unadjusted.csv")
nw84.to_csv(DDIR / "f7_84_three_ways.csv")

## CELL 7: CHECKPOINT and run log

In [ ]:
checks = {
    "combined_five_seeds": len(COMB) == 5,
    "capture_identical_to_reference": CAPTURE_EXACT,
    "reference_passes_all_checks": REF_PASSES_ALL,
    "detection_matrix_written": (DDIR / "detection_matrix.csv").exists(),
}
print(pd.Series(checks).to_string())
log = {"notebook": "ARTICLE2_NB4_Combined_and_Diagnostics", "purpose": "Article 2 only", "root": str(ROOT),
       "run_at": datetime.now().isoformat(timespec="seconds"), "versions": VERSIONS, "seeds": SEEDS,
       "c6": {"days": dg.C6_DAYS, "seed": dg.C6_SEED, "tolerance": dg.C6_TOL},
       "detection_matrix": matrix.to_dict(), "reference_passes_all": REF_PASSES_ALL, "missed": missed, "checks": checks}
with open(OUT / "logs" / "ARTICLE2_NB4_run_log.json", "w") as f:
    json.dump(log, f, indent=2, default=str)
soft = [k for k in ["reference_passes_all_checks"] if not checks[k]]
hard = [k for k, v in checks.items() if not v and k not in soft]
if soft:
    print("Needs a decision and a deviation-log entry (not a stop):", soft)
assert not hard, f"CHECKPOINT FAILED: {hard}"
print()
print("Hard checks: ALL PASSED")
print("Please share the outputs of CELLS 2, 3 (both parts), 4, 5, 6 and 7.")